# Final study: reproducible analysis

Run from the project root. The bundled simulation outcomes are sufficient for model evaluation. EnergyPlus is required only to regenerate physical simulations. Units throughout: cooling thermal demand, not electricity.

In [1]:
from pathlib import Path
import sys, subprocess, json, csv
ROOT = Path.cwd()
if not (ROOT / 'results/simulation_results.csv').exists():
    raise RuntimeError('Open the notebook from the project root.')
sys.path.insert(0, str(ROOT / 'scripts'))
rows = list(csv.DictReader((ROOT / 'results/simulation_results.csv').open(encoding='utf-8-sig')))
from collections import Counter
print('Completed simulations:', len(rows))
print(Counter(r['group'] for r in rows))

Completed simulations: 1890
Counter({'train_rectangle': 648, 'train_family': 360, 'test_family_area': 240, 'sensitivity': 180, 'diagnostic_shape_only': 144, 'diagnostic_area_only': 108, 'test_area': 72, 'test_ratio_interpolation': 48, 'daylight_application': 36, 'test_ratio_extrapolation': 24, 'zoning_check': 18, 'circle_convergence': 12})


## Refit fixed models and regenerate held-out predictions
The six strategies stay together in validation groups. Rotationally equivalent cases are grouped. Selection uses training CV only. The post-hoc diagnostic set remains separate.

In [2]:
completed = subprocess.run([sys.executable, 'scripts/analyze.py'], cwd=ROOT, capture_output=True, text=True, check=True)
print(completed.stdout)

Evaluated strategy_mean
Evaluated interpolation
Evaluated ridge
Evaluated forest
Selected on training CV: ridge external n= 384



In [3]:
metrics = list(csv.DictReader((ROOT/'results/pooled_metrics.csv').open(encoding='utf-8-sig')))
for r in metrics:
    if r['subset']=='all':
        print(r['scheme'], r['model'], 'MAE=', round(float(r['mae']), 4))

grouped_cv strategy_mean MAE= 9.9713
grouped_cv interpolation MAE= 5.3769
grouped_cv ridge MAE= 0.7035
grouped_cv forest MAE= 1.0069
rectangle_transfer strategy_mean MAE= 5.5768
rectangle_transfer interpolation MAE= 1.4076
rectangle_transfer ridge MAE= 0.4765
rectangle_transfer forest MAE= 5.3605
expanded_transfer strategy_mean MAE= 5.4734
expanded_transfer interpolation MAE= 1.8899
expanded_transfer ridge MAE= 0.471
expanded_transfer forest MAE= 5.3605
whole_family_holdout strategy_mean MAE= 8.5362
whole_family_holdout interpolation MAE= 4.4294
whole_family_holdout ridge MAE= 1.2776
whole_family_holdout forest MAE= 1.7085
diagnostic_transfer strategy_mean MAE= 6.3903
diagnostic_transfer interpolation MAE= 2.4378
diagnostic_transfer ridge MAE= 0.3687
diagnostic_transfer forest MAE= 0.8578


In [4]:
effects = list(csv.DictReader((ROOT/'results/paired_metrics.csv').open(encoding='utf-8-sig')))
for r in effects:
    if r['model']=='ridge':
        print(r['scheme'], 'saving MAE:', round(float(r['saving_mae']), 4), 'reduction MAE (pp):', round(float(r['reduction_mae_pp']), 4))

diagnostic_transfer saving MAE: 0.8676 reduction MAE (pp): 0.6192
expanded_transfer saving MAE: 0.6384 reduction MAE (pp): 0.4529
grouped_cv saving MAE: 1.3521 reduction MAE (pp): 0.9531
rectangle_transfer saving MAE: 0.6842 reduction MAE (pp): 0.4898
whole_family_holdout saving MAE: 1.5577 reduction MAE (pp): 1.0652


In [5]:
choices = list(csv.DictReader((ROOT/'results/design_choices.csv').open(encoding='utf-8-sig')))
for r in choices:
    print(r['context'], 'unconstrained:', r['unconstrained_option'], 'screened:', r['screened_option'])
print('These thermal contexts occur in development data; zero regret is not independent recommendation validation.')

rect2_a900_o0_w0.2_0.6_base_70_1_single unconstrained: SN4023_d0.6 screened: SN7035_d0.6
rect2_a900_o0_w0.4_0.6_base_70_1_single unconstrained: SN4023_d0.6 screened: SN4023_d0.6
rect2_a900_o0_w0.6_0.6_base_70_1_single unconstrained: SN4023_d0.6 screened: SN4023_d0.6
rect2_a900_o45_w0.2_0.6_base_70_1_single unconstrained: SN4023_d0.6 screened: SN7035_d0.6
rect2_a900_o45_w0.4_0.6_base_70_1_single unconstrained: SN4023_d0.6 screened: SN4023_d0.6
rect2_a900_o45_w0.6_0.6_base_70_1_single unconstrained: SN4023_d0.6 screened: SN4023_d0.6
These thermal contexts occur in development data; zero regret is not independent recommendation validation.


In [6]:
completed = subprocess.run([sys.executable, 'scripts/verify.py'], cwd=ROOT, capture_output=True, text=True, check=True)
print(completed.stdout)

{
  "passed": true,
  "cases": 1890,
  "train_cases": 1008,
  "external_cases": 384,
  "paired_contexts": 315,
  "severe_errors": 0,
  "warnings": {
    "** Warning ** CheckUsedConstructions: There are 15 nominally unused constructions in input.": 1890,
    "** Warning ** ProcessScheduleInput: Schedule:Day:Interval=\"SCHEDULE DAY 3\", Blank Schedule Type Limits Name input -- will not be validated.": 18,
    "** Warning ** ProcessScheduleInput: Schedule:Day:Interval=\"SCHEDULE DAY 4\", Blank Schedule Type Limits Name input -- will not be validated.": 18,
    "** Warning ** ProcessScheduleInput: Schedule:Year=\"0.05 HOT WATER LATENT FRACTION\", Blank Schedule Type Limits Name input -- will not be validated.": 18,
    "** Warning ** ProcessScheduleInput: Schedule:Year=\"0.2 HOT WATER SENSIBLE FRACTION\", Blank Schedule Type Limits Name input -- will not be validated.": 18,
    "** Warning ** ProcessScheduleInput: Schedule:Constant=\"ALWAYS OFF DISCRETE\", Blank Schedule Type Limits Name i

## Interpretation
Do not infer real-building accuracy from simulation consistency. The 9-point daylight screen is not sDA or compliance. Numerical geometric distance is not an uncertainty interval. The benchmark includes separate sequential EnergyPlus runs; reproduce it with scripts/benchmark.py and EnergyPlus 24.2.

## Independent November follow-up
30 new simulations remain separate from the original 1,890. Predictions were frozen before these runs. Both selectors use simulated daylight feasibility; the simple rule is a required comparator.

In [7]:
completed = subprocess.run([sys.executable, 'scripts/final_validation.py', 'analyze'], cwd=ROOT, capture_output=True, text=True, check=True)
print(completed.stdout)
print((ROOT/'results/final_validation/decisions.csv').read_text(encoding='utf-8-sig'))

{
  "cases": 30,
  "severe_errors": 0,
  "training_unchanged": true,
  "metrics": {
    "independent_L600": {
      "n": 6,
      "mae": 0.6293002453341169,
      "p95": 1.1765911791072554,
      "max": 1.375546952663484
    },
    "independent_daylight": {
      "n": 24,
      "mae": 0.4683100435377388,
      "p95": 0.9884039768716384,
      "max": 1.2351202536369499
    }
  },
  "daylight_contexts": 4,
  "model_total_regret": 0.0,
  "rule_total_regret": 0.0,
  "pairwise_difference_mae": 0.5217927627935433,
  "pairwise_ordering_accuracy": 0.9733333333333334,
  "limitations": "Four rectangle daylight contexts; one L600 thermal context; follow-up design after earlier results; simulated daylight provided to both selectors; no empirical validation."
}

context,feasible,simulation_best,model_choice,rule_choice,model_regret,rule_regret,note
rect1.5_a850_o30_w0.2_0.6_base_70_1_single,4,SN7035/0.6,SN7035/0.6,SN7035/0.6,0.0,0.0,Both methods receive the same simulated daylight feasibility; no l

## Nonrectangular daylight extension
Five shapes at 850 m² and 30°, with WWR 20% and 40%. There are 60 primary cases and 60 repetitions with a finer sampling grid. The latter are not independent physical contexts. Frozen predictions and sampling revisions are retained; neither follow-up is used for fitting or tuning.

In [8]:
completed = subprocess.run([sys.executable, 'scripts/nonrect_daylight.py', 'analyze'], cwd=ROOT, capture_output=True, text=True, check=True)
print(completed.stdout)
print((ROOT/'results/nonrect_daylight/decisions.csv').read_text(encoding='utf-8-sig'))

{
  "primary_cases": 60,
  "audit_cases": 60,
  "independent_contexts": 10,
  "total_project_simulations": 2040,
  "primary_mae": 0.37545299242204305,
  "severe_errors": 0,
  "no_training_or_tuning": true,
  "feasible_contexts": 10,
  "choices_changed_by_daylight": 4,
  "model_correct": 10,
  "rule_correct": 9,
  "model_total_regret": 0,
  "rule_total_regret": 0.20785123540734673,
  "max_grid_difference_pp": 1.8207701476079308,
  "mean_grid_difference_pp": 0.27382698565723,
  "grid_feasibility_flips": 0,
  "grid_choice_changes": 0,
  "max_cooling_grid_difference": 0.0,
  "strata": [
    {
      "shape": "Lwide",
      "cases": 12,
      "mae": 0.5867675465379243,
      "max_error": 1.5980020160450863,
      "daylight_changed_choices": 0,
      "model_correct": 2,
      "rule_correct": 2,
      "max_grid_difference_pp": 1.4089470692475459,
      "mean_grid_difference_pp": 0.3433328633071338,
      "feasibility_flips": 0,
      "choice_grid_changes": 0
    },
    {
      "shape": "triang